In [1]:
import numpy as np 
import pandas as pd

In [2]:
# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

# Feature columns:
# keep everything except datetime and target columns
df = pd.read_csv('all_feature.csv')
target_cols = ["target_pm25", "target_pm10", "target_aqi", "target_temp"]
feature_cols = [c for c in df.columns if c not in (["datetime"] + target_cols)]

# Input and output matrices
X = df[feature_cols]
y = df[target_cols]

# Time-based split (do NOT shuffle for time-series data)
split_index = int(len(df) * 0.80)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Train rows:", len(X_train))
print("Test rows :", len(X_test))
print("Features  :", X_train.shape[1])

Train rows: 2054
Test rows : 514
Features  : 72


In [3]:
# ============================================================
# RANDOM FOREST: WITHOUT STANDARDIZATION
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Helper: adjusted R2
# ------------------------------------------------------------
def adjusted_r2(r2, n, p):
    return 1 - ((1 - r2) * (n - 1)) / (n - p - 1)

# ------------------------------------------------------------
# Helper: evaluation
# ------------------------------------------------------------
def evaluate_model(model_name, y_true, y_pred, n_features):
    print(f"\n{model_name}")
    print("=" * len(model_name))

    targets = ["PM2.5", "PM10", "AQI", "Temperature"]

    for i, target in enumerate(targets):
        y_t = y_true.iloc[:, i]
        y_p = y_pred[:, i]

        mae = mean_absolute_error(y_t, y_p)
        rmse = np.sqrt(mean_squared_error(y_t, y_p))
        r2 = r2_score(y_t, y_p)
        adj = adjusted_r2(r2, len(y_t), n_features)

        print(f"\n{target}")
        print(f"MAE         : {mae:.3f}")
        print(f"RMSE        : {rmse:.3f}")
        print(f"R2          : {r2:.3f}")
        print(f"Adjusted R2 : {adj:.3f}")

# ------------------------------------------------------------
# Random Forest model
# ------------------------------------------------------------
rf_base = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_model = MultiOutputRegressor(rf_base, n_jobs=-1)

print("Training Random Forest (without standardization)...")
rf_model.fit(X_train, y_train)
print("Training completed.")

rf_pred = rf_model.predict(X_test)

evaluate_model("Random Forest (No Standardization)", y_test, rf_pred, X_train.shape[1])

rf_pred_df = pd.DataFrame(
    rf_pred,
    columns=["pred_pm25", "pred_pm10", "pred_aqi", "pred_temp"]
)

print("\nSample predictions (RF no scaling):")
print(rf_pred_df.head())

Training Random Forest (without standardization)...
Training completed.

Random Forest (No Standardization)

PM2.5
MAE         : 5.380
RMSE        : 8.180
R2          : 0.910
Adjusted R2 : 0.896

PM10
MAE         : 8.367
RMSE        : 13.177
R2          : 0.896
Adjusted R2 : 0.879

AQI
MAE         : 8.456
RMSE        : 12.282
R2          : 0.933
Adjusted R2 : 0.922

Temperature
MAE         : 1.475
RMSE        : 2.093
R2          : 0.944
Adjusted R2 : 0.935

Sample predictions (RF no scaling):
   pred_pm25  pred_pm10    pred_aqi  pred_temp
0  51.235364  56.289422  133.904709  53.760183
1  60.852598  62.295765  145.239035  53.939548
2  53.180883  59.322304  138.185191  52.820502
3  51.904924  59.393271  136.149287  52.105287
4  65.459358  71.423750  154.494190  52.177896


In [4]:
# ============================================================
# RANDOM FOREST: WITH STANDARDIZATION
# ============================================================

# Note:
# Scaling is usually NOT needed for Random Forest.
# This block is only for comparison.

rf_scaled_base = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_scaled_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", MultiOutputRegressor(rf_scaled_base, n_jobs=-1))
])

print("\nTraining Random Forest (with standardization)...")
rf_scaled_model.fit(X_train, y_train)
print("Training completed.")

rf_scaled_pred = rf_scaled_model.predict(X_test)

evaluate_model("Random Forest (With Standardization)", y_test, rf_scaled_pred, X_train.shape[1])

rf_scaled_pred_df = pd.DataFrame(
    rf_scaled_pred,
    columns=["pred_pm25", "pred_pm10", "pred_aqi", "pred_temp"]
)

print("\nSample predictions (RF with scaling):")
print(rf_scaled_pred_df.head())


Training Random Forest (with standardization)...
Training completed.

Random Forest (With Standardization)

PM2.5
MAE         : 5.375
RMSE        : 8.166
R2          : 0.911
Adjusted R2 : 0.896

PM10
MAE         : 8.376
RMSE        : 13.196
R2          : 0.896
Adjusted R2 : 0.879

AQI
MAE         : 8.436
RMSE        : 12.270
R2          : 0.933
Adjusted R2 : 0.923

Temperature
MAE         : 1.476
RMSE        : 2.093
R2          : 0.944
Adjusted R2 : 0.935

Sample predictions (RF with scaling):
   pred_pm25  pred_pm10    pred_aqi  pred_temp
0  51.111009  56.697854  133.878993  53.750772
1  61.067850  62.395647  145.125356  53.933471
2  53.025839  59.427494  138.254306  52.824596
3  51.862224  59.644509  135.971302  52.109927
4  65.685213  71.352451  154.596989  52.196393
